# BigQuery Basics Hands-On Lab — gcloud & bq CLI
### GCP Training Program — Day 1, Module 1: Storage & Ingestion (BigQuery Basics)

**What this notebook covers:** simple SQL queries, table joins, views, materialized views,
partitioning, and clustering — all executed through the **`bq` command-line tool** (the CLI is
included in the Cloud SDK), with `gcloud` used only for project configuration and authentication.
No Python client library anywhere in this notebook.

## The Public Dataset: Austin Bikeshare
We'll use **`bigquery-public-data.austin_bikeshare`**, Google's public mirror of the City of
Austin's B-cycle bike-share program data. It's a good teaching dataset because it's small enough
to query cheaply in a live class, but has exactly the shape you need to demonstrate every concept
on the agenda:

- **`bikeshare_trips`** (~1.49 million rows) — one row per bike trip: `trip_id`, `subscriber_type`
  (e.g. Membership vs Single Trip), `bike_id`, `start_time`, `start_station_id`/`start_station_name`,
  `end_station_id`/`end_station_name`, `duration_minutes`. This is our **fact table** — it has a
  natural timestamp column (`start_time`) perfect for partitioning, and a station ID perfect for
  joins and clustering.
- **`bikeshare_stations`** — one row per physical docking station: `station_id`, `name`, `status`
  (active/closed), `council_district`, `number_of_docks`. This is our **dimension table** — small,
  slowly-changing, and the natural join partner for `bikeshare_trips`.

**Caveat for the class:** public dataset schemas can change column names over time. Section 3
below prints the live schema of both tables with `bq show` — if a column name here doesn't match
what you see printed, trust the live schema and adjust the queries accordingly.

**Cost note:** storage for `bigquery-public-data` tables is free — Google hosts it. *Querying* it
still counts against your project's on-demand query bytes (or your monthly free tier). Every query
section below shows how to check estimated bytes with `--dry_run` before running the real thing —
a habit worth teaching regardless of dataset.

**This notebook is fully self-contained.** Just run the cells top to bottom:
1. The **Authenticate** cell will pop up a Google sign-in flow.
2. The **Configure** cell will *ask you* for your Project ID and BigQuery location, and creates a
   working dataset for you automatically.
3. Everything after that runs against your own project.


## 1. Authenticate This Session
Sign in with the Google account that has access to your GCP project. This also configures the
`gcloud`/`bq` CLIs for the rest of this notebook.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated in Colab. This also configures gcloud and bq for this session.")
else:
    print("Not running in Colab — assuming gcloud Application Default Credentials are set up.")
    print("If this is your first time, run in a terminal:")
    print("  gcloud auth login")
    print("  gcloud auth application-default login")

## 2. Configure Your Project & Create a Working Dataset
Enter your own values when prompted below. A dataset is created automatically to hold every view,
materialized view, and partitioned table this notebook builds — it gets a timestamp suffix so
re-running this notebook never collides with a previous run.

In [ ]:
import time
import os

PROJECT_ID = input("Enter your GCP Project ID: ").strip()

BQ_LOCATION = input("Enter your BigQuery location [default: US]: ").strip()
if not BQ_LOCATION:
    BQ_LOCATION = "US"  # austin_bikeshare lives in the US multi-region

DATASET_NAME = f"bq_training_demo_{int(time.time())}"

# Export as real shell environment variables — the %%bash query cells below reference these
# as $PROJECT_ID / $DATASET_NAME / $BQ_LOCATION rather than Python string interpolation.
os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["DATASET_NAME"] = DATASET_NAME
os.environ["BQ_LOCATION"] = BQ_LOCATION

print("Project:       ", PROJECT_ID)
print("BQ location:   ", BQ_LOCATION)
print("Working dataset:", DATASET_NAME)

In [ ]:
!gcloud config set project {PROJECT_ID}
!bq mk --dataset --location={BQ_LOCATION} {PROJECT_ID}:{DATASET_NAME}

## 3. Explore the Public Dataset
### 3.1 Confirm the Live Schema
Print the real, current schema for both tables before writing any queries against them — the
safest way to avoid a query failing on a renamed or missing column.

In [ ]:
!bq show --format=prettyjson bigquery-public-data:austin_bikeshare.bikeshare_trips | head -60

In [ ]:
!bq show --format=prettyjson bigquery-public-data:austin_bikeshare.bikeshare_stations | head -60

### 3.2 Row Counts & Table Size
`bq show` (without `--format=prettyjson`) prints a compact summary including row count and size on
disk — useful context before running any query against a table you haven't used before.

In [ ]:
!bq show bigquery-public-data:austin_bikeshare.bikeshare_trips
!bq show bigquery-public-data:austin_bikeshare.bikeshare_stations

### 3.3 Sample Rows From Each Table

In [ ]:
%%bash
bq query --use_legacy_sql=false --format=pretty <<'EOF'
SELECT *
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
LIMIT 5
EOF

In [ ]:
%%bash
bq query --use_legacy_sql=false --format=pretty <<'EOF'
SELECT *
FROM `bigquery-public-data.austin_bikeshare.bikeshare_stations`
LIMIT 5
EOF

## 4. Simple SQL Queries
### 4.1 Total Trip Count
**What this query does:** `COUNT(*)` counts every row in the table — one row per bike trip — with
no filtering at all. This is the simplest possible aggregate query and a good sanity check before
anything more complex.

In [ ]:
%%bash
bq query --use_legacy_sql=false --format=pretty <<'EOF'
SELECT COUNT(*) AS total_trips
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
EOF

### 4.2 Trip Duration Statistics
**What this query does:** `AVG`, `MAX`, and `MIN` are aggregate functions computed across every
matching row. `ROUND(..., 2)` formats the average to 2 decimal places. The `WHERE` clause filters
out `NULL` or zero/negative durations first — a data-quality guard that's common in real pipelines,
since a handful of bad readings can otherwise skew an average.

In [ ]:
%%bash
bq query --use_legacy_sql=false --format=pretty <<'EOF'
SELECT
  ROUND(AVG(duration_minutes), 2) AS avg_duration_minutes,
  MAX(duration_minutes) AS max_duration_minutes,
  MIN(duration_minutes) AS min_duration_minutes
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
WHERE duration_minutes IS NOT NULL AND duration_minutes > 0
EOF

### 4.3 Top 10 Busiest Start Stations
**What this query does:** `GROUP BY start_station_name` collapses all trip rows into one row per
station name. `COUNT(*)` then counts how many trip rows fell into each group. `ORDER BY trip_count
DESC` sorts busiest-first, and `LIMIT 10` keeps only the top 10 — a classic "top N" query pattern.

In [ ]:
%%bash
bq query --use_legacy_sql=false --format=pretty <<'EOF'
SELECT
  start_station_name,
  COUNT(*) AS trip_count
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
WHERE start_station_name IS NOT NULL
GROUP BY start_station_name
ORDER BY trip_count DESC
LIMIT 10
EOF

### 4.4 Trips by Subscriber Type, With Percentage Share
**What this query does:** groups trips by `subscriber_type` (e.g. Membership vs Single Trip), then
uses a **window function** — `SUM(COUNT(*)) OVER ()` — to compute the grand total *across all
groups* without a second query, so each row can show its percentage of the overall total. The
empty `OVER ()` means "apply this aggregate across the entire result set," not just the current
group — this is the key difference between a window function and a normal `GROUP BY` aggregate.

In [ ]:
%%bash
bq query --use_legacy_sql=false --format=pretty <<'EOF'
SELECT
  subscriber_type,
  COUNT(*) AS trip_count,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_total
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
GROUP BY subscriber_type
ORDER BY trip_count DESC
EOF

## 5. Table Joins
### 5.1 INNER JOIN — Enrich Trips With Station Details
**What this query does:** joins the fact table (`t` = trips) to the dimension table (`s` = stations)
on `t.start_station_id = s.station_id`. An **INNER JOIN** keeps only rows where a match exists on
both sides — any trip whose station ID doesn't exist in the current stations table is silently
dropped (Section 5.3 shows how to find exactly those dropped rows).

In [ ]:
%%bash
bq query --use_legacy_sql=false --format=pretty <<'EOF'
SELECT
  t.trip_id,
  t.start_time,
  t.start_station_name,
  s.status AS start_station_status,
  s.council_district,
  t.duration_minutes
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips` AS t
INNER JOIN `bigquery-public-data.austin_bikeshare.bikeshare_stations` AS s
  ON t.start_station_id = s.station_id
LIMIT 20
EOF

### 5.2 Aggregation Across a Join — Demand vs. Capacity per Station
**What this query does:** joins stations to trips, then groups by station attributes
(`s.name`, `s.number_of_docks`) to compute trip volume and average duration per station. This is a
classic **fact-to-dimension join with aggregation** — the kind of query that answers "which
stations are busiest relative to how many docks they have," useful for capacity planning.

In [ ]:
%%bash
bq query --use_legacy_sql=false --format=pretty <<'EOF'
SELECT
  s.name AS station_name,
  s.number_of_docks,
  COUNT(t.trip_id) AS trip_count,
  ROUND(AVG(t.duration_minutes), 2) AS avg_duration_minutes
FROM `bigquery-public-data.austin_bikeshare.bikeshare_stations` AS s
INNER JOIN `bigquery-public-data.austin_bikeshare.bikeshare_trips` AS t
  ON s.station_id = t.start_station_id
GROUP BY s.name, s.number_of_docks
ORDER BY trip_count DESC
LIMIT 10
EOF

### 5.3 LEFT JOIN — Find Trips With No Matching Station
**What this query does:** a **LEFT JOIN** keeps every row from the left table (`trips`) regardless
of whether a match exists on the right (`stations`) — unmatched rows just get `NULL` for every
column from `stations`. Filtering `WHERE s.station_id IS NULL` isolates exactly the trips that
*would have been silently dropped* by the INNER JOIN in Section 5.1 — station IDs that have since
been renamed, decommissioned, or renumbered. This "find the orphans" pattern is one of the most
common real-world uses of LEFT JOIN.

In [ ]:
%%bash
bq query --use_legacy_sql=false --format=pretty <<'EOF'
SELECT
  t.start_station_id,
  t.start_station_name,
  COUNT(*) AS orphaned_trip_count
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips` AS t
LEFT JOIN `bigquery-public-data.austin_bikeshare.bikeshare_stations` AS s
  ON t.start_station_id = s.station_id
WHERE s.station_id IS NULL
GROUP BY t.start_station_id, t.start_station_name
ORDER BY orphaned_trip_count DESC
LIMIT 20
EOF

## 6. Views
### 6.1 Create a View
**What this does:** `CREATE VIEW` stores the *query itself*, not its results — a view has no data
of its own. Every time someone queries it, BigQuery re-runs the underlying `SELECT` against the
live base tables. We're wrapping the station-summary join from Section 5.2 so analysts can query
`station_trip_summary_view` directly instead of re-writing that join every time.

`--dataset_id=$DATASET_NAME` below sets the *default* dataset for this query, so the bare view name
(no project/dataset prefix) is created inside your own working dataset.

In [ ]:
%%bash
bq query --use_legacy_sql=false --dataset_id=$DATASET_NAME --location=$BQ_LOCATION <<'EOF'
CREATE VIEW station_trip_summary_view AS
SELECT
  s.station_id,
  s.name AS station_name,
  s.number_of_docks,
  COUNT(t.trip_id) AS trip_count,
  ROUND(AVG(t.duration_minutes), 2) AS avg_duration_minutes
FROM `bigquery-public-data.austin_bikeshare.bikeshare_stations` AS s
LEFT JOIN `bigquery-public-data.austin_bikeshare.bikeshare_trips` AS t
  ON s.station_id = t.start_station_id
GROUP BY s.station_id, s.name, s.number_of_docks
EOF

### 6.2 Query the View Like a Table

In [ ]:
%%bash
bq query --use_legacy_sql=false --format=pretty --dataset_id=$DATASET_NAME --location=$BQ_LOCATION <<'EOF'
SELECT *
FROM station_trip_summary_view
ORDER BY trip_count DESC
LIMIT 10
EOF

### 6.3 Views: Advantages & Trade-offs (talking points for the class)
**Advantages:**
- **Reusable abstraction** — hides a complex join behind a simple name; analysts don't need to
  know the underlying schema.
- **Always live** — since it re-runs the query each time, a view can never show stale data.
- **No storage cost** — a view stores SQL text, not rows, so there's nothing to pay to keep it
  around.
- **Access control** — an *authorized view* can expose a filtered/aggregated slice of a table to
  someone without granting them access to the underlying base table (used heavily in Day 2's
  governance module).

**Trade-offs:**
- **No performance benefit** — every query against the view re-executes the full underlying join
  and aggregation. If the base tables are huge, querying the view is exactly as expensive as
  running the raw SQL yourself.
- **No incremental caching** — unlike a materialized view (Section 7), there's no precomputation
  at all.

## 7. Materialized Views
### 7.1 Create a Materialized View
**What this does:** unlike a regular view, `CREATE MATERIALIZED VIEW` actually **precomputes and
stores** the result, then keeps it incrementally refreshed as the base table changes — queries
against it read the cached result instead of recomputing the aggregation from scratch every time.

**Caveat for the class:** materialized views support a restricted subset of SQL (no
non-deterministic functions like `CURRENT_TIMESTAMP()`, limited join types, can't reference other
views) — check current BigQuery documentation for the exact supported surface, since it has
expanded across releases.

In [ ]:
%%bash
bq query --use_legacy_sql=false --dataset_id=$DATASET_NAME --location=$BQ_LOCATION <<'EOF'
CREATE MATERIALIZED VIEW daily_station_trip_counts_mv AS
SELECT
  DATE(start_time) AS trip_date,
  start_station_name,
  COUNT(*) AS trip_count,
  ROUND(AVG(duration_minutes), 2) AS avg_duration_minutes
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
WHERE start_station_name IS NOT NULL
GROUP BY trip_date, start_station_name
EOF

### 7.2 Query the Materialized View Directly

In [ ]:
%%bash
bq query --use_legacy_sql=false --format=pretty --dataset_id=$DATASET_NAME --location=$BQ_LOCATION <<'EOF'
SELECT *
FROM daily_station_trip_counts_mv
WHERE trip_date = '2016-01-01'
ORDER BY trip_count DESC
LIMIT 10
EOF

### 7.3 Cost Comparison: Materialized View vs. Re-Running the Aggregation
**What we're comparing:** the same logical query — daily trip counts per station for a single day —
run two ways. `--dry_run` doesn't execute the query or cost anything; it prints the **estimated
bytes that would be processed**, which is exactly what determines on-demand query cost. Compare the
two numbers printed below: the materialized view reads its small, precomputed result; the raw
query re-scans and re-aggregates the full base table.

**Query A — against the materialized view:**

In [ ]:
%%bash
bq query --use_legacy_sql=false --dry_run --dataset_id=$DATASET_NAME --location=$BQ_LOCATION <<'EOF'
SELECT *
FROM daily_station_trip_counts_mv
WHERE trip_date = '2016-01-01'
ORDER BY trip_count DESC
LIMIT 10
EOF

**Query B — the same aggregation run directly against the base table:**

In [ ]:
%%bash
bq query --use_legacy_sql=false --dry_run --dataset_id=$DATASET_NAME --location=$BQ_LOCATION <<'EOF'
SELECT
  DATE(start_time) AS trip_date,
  start_station_name,
  COUNT(*) AS trip_count,
  ROUND(AVG(duration_minutes), 2) AS avg_duration_minutes
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
WHERE start_station_name IS NOT NULL AND DATE(start_time) = '2016-01-01'
GROUP BY trip_date, start_station_name
ORDER BY trip_count DESC
LIMIT 10
EOF

### 7.4 Materialized Views: Advantages & Limitations (talking points for the class)
**Advantages:**
- **Precomputed & incrementally refreshed** — BigQuery automatically keeps the stored result in
  sync with the base table, refreshing only the changed portion rather than recomputing everything.
- **Smart query routing** — even a query against the *base table* can transparently be rewritten by
  BigQuery's optimizer to read from a matching materialized view instead, without the analyst
  writing any different SQL.
- **Lower cost & latency** for repeated dashboard-style queries over the same aggregation.

**Limitations:**
- Restricted SQL surface (see caveat above) — not every view can be materialized.
- Refresh has some latency — a materialized view is near-real-time, not instantaneous, unlike a
  regular view which is always 100% live.
- Storage cost — unlike a regular view, a materialized view does store data and gets billed for it.

## 8. Partitioning
### 8.1 Create a Partitioned Copy of the Trips Table
**What this does:** `PARTITION BY DATE(start_time)` splits the table into one physical partition
per calendar day. A query that filters on `start_time`/`DATE(start_time)` can then skip every
partition outside the filtered range entirely — it never reads that data off disk at all. This is
**partition pruning**, and it's the single biggest lever for cutting query cost on a time-series
table.

We filter out `NULL` start times first — BigQuery puts NULL-partitioning-key rows into a special
`__NULL__` partition, which is rarely useful and easy to avoid up front.

In [ ]:
%%bash
bq query --use_legacy_sql=false --dataset_id=$DATASET_NAME --location=$BQ_LOCATION <<'EOF'
CREATE TABLE trips_partitioned
PARTITION BY DATE(start_time) AS
SELECT *
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
WHERE start_time IS NOT NULL
EOF

### 8.2 See Partition Pruning in Action: Bytes Scanned, With vs. Without Partitioning
Same filter — a single day — run against two tables: the original public table (no partitioning),
and our own partitioned copy. Compare the **Total bytes processed** line each `--dry_run` prints.

**Query A — against the original, unpartitioned public table (must scan every row to check every
date):**

In [ ]:
%%bash
bq query --use_legacy_sql=false --dry_run <<'EOF'
SELECT COUNT(*) AS trips_on_day
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
WHERE DATE(start_time) = '2016-06-15'
EOF

**Query B — against our partitioned copy (BigQuery prunes straight to the one matching
partition):**

In [ ]:
%%bash
bq query --use_legacy_sql=false --dry_run --dataset_id=$DATASET_NAME --location=$BQ_LOCATION <<'EOF'
SELECT COUNT(*) AS trips_on_day
FROM trips_partitioned
WHERE DATE(start_time) = '2016-06-15'
EOF

### 8.3 Partitioning: Types & Best Practices (talking points for the class)
**Partition types available in BigQuery:**
- **Time-unit column partitioning** (what we just did) — partitioned by a `DATE`/`TIMESTAMP`/
  `DATETIME` column, at daily/hourly/monthly/yearly granularity.
- **Ingestion-time partitioning** — BigQuery partitions automatically by the time each row was
  loaded, exposed via the pseudocolumn `_PARTITIONTIME`, useful when there's no natural event-time
  column in the source data.
- **Integer-range partitioning** — partitions by ranges of an integer column (e.g. a numeric
  customer ID or zip code) instead of a date.

**Best practices worth mentioning live:**
- `require_partition_filter=true` can be set on a partitioned table to *force* every query to
  include a partition filter — a good safety net against an analyst accidentally full-scanning
  a multi-terabyte table.
- There's a maximum number of partitions per table (in the low thousands) — very high-cardinality
  partitioning (e.g. by exact timestamp) will fail; daily or hourly granularity is almost always
  the right choice for typical event data.

## 9. Clustering
### 9.1 Create a Partitioned *and* Clustered Copy
**What this does:** clustering sorts the data *within* each partition by the given column(s) — here,
`start_station_id`. A query that filters or aggregates on the cluster column can then skip blocks
of data within a partition that don't match, on top of whatever partition pruning already achieved.
Partitioning narrows down to a day; clustering narrows down further within that day.

In [ ]:
%%bash
bq query --use_legacy_sql=false --dataset_id=$DATASET_NAME --location=$BQ_LOCATION <<'EOF'
CREATE TABLE trips_partitioned_clustered
PARTITION BY DATE(start_time)
CLUSTER BY start_station_id AS
SELECT *
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
WHERE start_time IS NOT NULL
EOF

### 9.2 Compare a Station-Filtered Query: Partitioned-Only vs. Partitioned+Clustered
**Caveat for the class:** unlike partition pruning, clustering's benefit doesn't always show up
clearly in `--dry_run`'s bytes estimate — the estimator can undercount block-level pruning. Treat
the numbers below as indicative, and for a ground-truth comparison, check the *actual* bytes
billed on a completed job with `bq show --format=prettyjson -j <job_id>` after really running the
query (not just dry-running it).

In [ ]:
%%bash
bq query --use_legacy_sql=false --dry_run --dataset_id=$DATASET_NAME --location=$BQ_LOCATION <<'EOF'
SELECT COUNT(*) AS trips
FROM trips_partitioned
WHERE DATE(start_time) = '2016-06-15' AND start_station_id = 1006
EOF

In [ ]:
%%bash
bq query --use_legacy_sql=false --dry_run --dataset_id=$DATASET_NAME --location=$BQ_LOCATION <<'EOF'
SELECT COUNT(*) AS trips
FROM trips_partitioned_clustered
WHERE DATE(start_time) = '2016-06-15' AND start_station_id = 1006
EOF

## 10. Cleanup — Delete Views, Tables & the Working Dataset
Run this **last**, once every demo above has been shown, so the class sees teardown too. `bq rm -f`
suppresses the confirmation prompt; `-r` on the dataset removal recursively deletes anything left
inside it.

In [ ]:
!bq rm -f -t {PROJECT_ID}:{DATASET_NAME}.daily_station_trip_counts_mv
!bq rm -f -t {PROJECT_ID}:{DATASET_NAME}.station_trip_summary_view
!bq rm -f -t {PROJECT_ID}:{DATASET_NAME}.trips_partitioned
!bq rm -f -t {PROJECT_ID}:{DATASET_NAME}.trips_partitioned_clustered
!bq rm -f -r -d {PROJECT_ID}:{DATASET_NAME}
print(f"Dataset {DATASET_NAME} and all contents deleted.")